In [1]:
import graphite


In [2]:
dataset = graphite.Dataset(source='washington',
                           level='line',
                           training_ratio=None,
                           validation_ratio=None,
                           test_ratio=None,
                           lazy_mode=False,
                           seed=42)
print(dataset)



Dataset Configuration

Source                  washington
Recognition Level       line
Training Ratio          -
Validation Ratio        -
Test Ratio              -
Padding Value           255
Lazy Mode               False
Seed                    42


Dataset Information

Total Size              656

Training Size           325
Validation Size         168
Test Size               163

Corpus Length           21078
Corpus                  Recruits . They are to see that each man distinguishes desired not to plan any work , which requires [...]

Charset Length          67
Charset                  '(),.0123456789:;ABCDEFGHIJKLMNOPQRSTVWYabcdefghijklmnopqrstuvwxyz

Min Text Length         4
Min Text                mand

Max Text Length         62
Max Text                of it , hoping that it will correspond with your own opinion .

Min Rows                1
Max Rows                1

Min Columns             4
Max Columns             62



In [3]:
augmentor = graphite.Augmentor(  # elastic_transform=[0.99, 43, 0.75],
    erosion=[0.99, 5, 1],
    dilation=[0.99, 2, 1],
    #    mixup=[0.99, 0.3, 1],
    #    perspective_transform=[0.99, 0.4],
    #    salt_and_pepper=[0.99, 0.3],
    #    gaussian_blur=[0.99, 11, 1],
    #    shearing=[0.99, 30],
    scaling=[0.99, 0.05],
    rotation=[0.99, 0.5],
    translation=[0.99, 0.05, 0.05],
    seed=42)
print(augmentor)



Augmentor Configuration

Elastic Transform       None
Erosion                 [0.99, 5, 1]
Dilation                [0.99, 2, 1]

Mixup                   None
Perspective Transform   None

Salt and Pepper Noise   None
Gaussian Blur           None

Shearing                None
Scaling                 [0.99, 0.05]
Rotation                [0.99, 0.5]
Translation             [0.99, 0.05, 0.05]

Padding Value           255
Augmentation Disabled   False
Seed                    42



In [4]:
model = graphite.Model(network='flor', tokenizer=dataset.tokenizer, seed=42)
model.compile(learning_rate=1e-3, run_index=None)

print(model)



Model Configuration

Network                     flor
Tokenizer Shape             (1, 64, 70)
Padding Value               255
Seed                        42

Optimizer                   RMSprop
Learning Rate               0.0010000000474974513

Summary

Model: "model"
_________________________________________________________________
Layer (type)                Output Shape              Param #
input_1 (InputLayer)        [(None, None, None, 1)]   0

input_processing (InputPro  (None, 1024, 128, 1)      0
cessing)

conv2d (Conv2D)             (None, 512, 64, 16)       160

p_re_lu (PReLU)             (None, 512, 64, 16)       16

batch_normalization (Batch  (None, 512, 64, 16)       112
Normalization)

full_gated_conv2d (FullGat  (None, 512, 64, 16)       4640
edConv2D)

conv2d_1 (Conv2D)           (None, 512, 64, 32)       4640

p_re_lu_1 (PReLU)           (None, 512, 64, 32)       32

batch_normalization_1 (Bat  (None, 512, 64, 32)       224
chNormalization)

full_gated_conv2d_1 (Ful

In [5]:
train_data, train_steps = dataset.get_generator(dataset.training, batch_size=16, augmentor=augmentor)
valid_data, valid_steps = dataset.get_generator(dataset.validation, batch_size=16, augmentor=None)

model.fit(training_data=train_data,
          training_steps=train_steps,
          validation_data=valid_data,
          validation_steps=valid_steps,
          #   plateau_cooldown=0,
          #   plateau_factor=0.1,
          #   plateau_patience=10,
          #   patience=20,
          plateau_cooldown=15,
          plateau_factor=0.25,
          plateau_patience=15,
          patience=40,
          #   plateau_cooldown=20,
          #   plateau_factor=0.50,
          #   plateau_patience=20,
          #   patience=60,
          epochs=1000,
          verbose=1)


Epoch 1/1000
21/21 [==============================] - 17s 499ms/step - loss: 190.5467 - val_loss: 136.1190 - lr: 0.0010
Epoch 2/1000
21/21 [==============================] - 6s 302ms/step - loss: 137.6047 - val_loss: 135.3717 - lr: 0.0010
Epoch 3/1000
21/21 [==============================] - 6s 283ms/step - loss: 132.2030 - val_loss: 128.4683 - lr: 0.0010
Epoch 4/1000
21/21 [==============================] - 6s 284ms/step - loss: 123.4910 - val_loss: 119.5147 - lr: 0.0010
Epoch 5/1000
21/21 [==============================] - 6s 278ms/step - loss: 115.5655 - val_loss: 114.9711 - lr: 0.0010
Epoch 6/1000
21/21 [==============================] - 6s 290ms/step - loss: 107.4976 - val_loss: 112.9156 - lr: 0.0010
Epoch 7/1000
21/21 [==============================] - 6s 285ms/step - loss: 103.3231 - val_loss: 102.5233 - lr: 0.0010
Epoch 8/1000
21/21 [==============================] - 6s 277ms/step - loss: 95.2022 - val_loss: 105.3934 - lr: 0.0010
Epoch 9/1000
21/21 [============================

In [ ]:
test_data, test_steps = dataset.get_generator(dataset.test, batch_size=16, augmentor=None)

predictions, probabilities = model.predict(test_data=test_data,
                                           test_steps=test_steps,
                                           top_paths=1,
                                           beam_width=30,
                                           ctc_decode=True,
                                           token_decode=True,
                                           verbose=1)

print(predictions)


In [ ]:

metrics, _ = model.evaluate(dataset.test,
                            predictions=predictions,
                            share_top_paths=True,
                            prediction_samples=10,
                            origin='vanilla')

print(metrics)


In [ ]:
# instruction = """
#     Correct any spelling mistakes, including accents, on each of the following lines.
#     Preserve slang, historical language, and original grammar.
#     Treat each line as an isolated piece of text.
#     Only change text if confident.
# """

# spelling = graphite.Spelling(spell_checker='openai', env_key='OPENAI_API_KEY')
# enhanced_predictions = spelling.enhance(predictions, instruction=instruction)

# enhanced_predictions


In [ ]:
spelling = graphite.Spelling(spell_checker='openai', env_key='OPENAI_API_KEY')
enhanced_predictions = spelling.enhance(predictions)

enhanced_predictions


In [ ]:
enhanced_metrics, _ = model.evaluate(dataset.test,
                                     predictions=enhanced_predictions,
                                     share_top_paths=True,
                                     prediction_samples=10,
                                     origin='openai')

print(enhanced_metrics)
